# 三大数据源市值字段对比(V3)

## 1. 导入与读取

In [22]:
import sys, os, warnings
warnings.filterwarnings("ignore")
import polars as pl
sys.path.insert(0, os.getcwd())
from my_utils.fun import DATA_ROOT_DIR
gm = pl.scan_parquet(os.path.join(DATA_ROOT_DIR, "gm_stock_all_data")).collect()
rq = pl.scan_parquet(os.path.join(DATA_ROOT_DIR, "rq_stock_all_data")).collect()
ts = pl.scan_parquet(os.path.join(DATA_ROOT_DIR, "ts_stock_all_data")).collect()
tb = pl.scan_parquet(os.path.join(DATA_ROOT_DIR, "ts_daily_basic")).collect()
gd = set(gm["trading_date"].unique().to_list())
rd = set(rq["trading_date"].unique().to_list())
td = set(ts["trading_date"].unique().to_list())
bd = set(tb["trading_date"].unique().to_list())
gm_rq = sorted(gd & rd)
ts_gm = sorted((td & bd) & gd)
print(f"TS:{len(td)}d GM:{len(gd)}d RQ:{len(rd)}d")
print(f"GM-RQ:{len(gm_rq)}d TS-GM:{len(ts_gm)}d")
print(f"TS-RQ:{len((td & bd) & rd)}d")

TS:1272d GM:1299d RQ:6d
GM-RQ:6d TS-GM:1272d
TS-RQ:0d


## 2. 单位确认

In [23]:
m = tb.filter(pl.col("trading_date").is_in(ts_gm))
m = m.join(gm.filter(pl.col("trading_date").is_in(ts_gm))
    .select(["code", "trading_date", "total_mv"]),
    on=["code", "trading_date"], suffix="_gm")
print(f"TS:万元(x1e4=元) GM/TS={m['total_mv_gm'].median()/m['total_mv'].median():.0f}")
m2 = gm.filter(pl.col("trading_date").is_in(gm_rq))
m2 = m2.join(rq.filter(pl.col("trading_date").is_in(gm_rq))
    .select(["code", "trading_date", "total_mv"]),
    on=["code", "trading_date"], suffix="_rq")
r2 = m2["total_mv"].median() / m2["total_mv_rq"].median()
print(f"RQ:元同GM GM/RQ={r2:.4f}")

TS:万元(x1e4=元) GM/TS=9942
RQ:元同GM GM/RQ=1.0007


## 3. 自由流通:逐股票对比TS(free_share) vs RQ(mv_A_free_float)


由于TS和RQ无共同交易日，桥接方法:
1. 对每只股票，在TS-GM重叠日计算 TS自由/GM流通 的平均比值
2. 对每只股票，在GM-RQ重叠日计算 GM流通/RQ自由 的平均比值
3. 乘起来 = TS自由/RQ自由


In [24]:
# TS侧: 每只股票 TS(free_share x close) / GM(流通) 的平均比值
tbf = tb.filter(pl.col("free_share").is_not_null() & pl.col("close").is_not_null())
tbf = tbf.with_columns(
    (pl.col("free_share") * pl.col("close") * 10000).alias("ff_yuan"))
tg = tbf.filter(pl.col("trading_date").is_in(ts_gm))
tg = tg.select(["code", "trading_date", "ff_yuan"])
tg = tg.join(gm.select(["code", "trading_date", "mv_A_free_float"]),
    on=["code", "trading_date"])
tg = tg.with_columns((pl.col("ff_yuan") / pl.col("mv_A_free_float")).alias("ratio"))
ts_avg = tg.group_by("code").agg(
    pl.col("ratio").mean().alias("ts_gm_ratio"),
    pl.col("ratio").count().alias("n_days"),
    pl.col("ff_yuan").mean().alias("avg_ts_ff"),
    pl.col("mv_A_free_float").mean().alias("avg_gm"))
print(f"TS侧有数据的股票:{ts_avg.height:,}")
tr = ts_avg["ts_gm_ratio"]
print(f"TS自由/GM流通 股票级比值分布:")
for p in [5,25,50,75,95]:
    print(f"  P{p}: {tr.quantile(p/100):.4f}")

TS侧有数据的股票:5,385
TS自由/GM流通 股票级比值分布:
  P5: 0.3461
  P25: 0.5473
  P50: 0.6948
  P75: 0.8197
  P95: 0.9753


In [25]:
# RQ侧: 每只股票 GM(流通) / RQ(自由) 的平均比值
gf = gm.filter(pl.col("trading_date").is_in(gm_rq))
gf = gf.join(rq.select(["code", "trading_date", "mv_A_free_float"]),
    on=["code", "trading_date"], suffix="_rq")
gf = gf.with_columns(
    (pl.col("mv_A_free_float") / pl.col("mv_A_free_float_rq")).alias("ratio"))
rq_avg = gf.group_by("code").agg(
    pl.col("ratio").mean().alias("gm_rq_ratio"),
    pl.col("ratio").count().alias("n_days"),
    pl.col("mv_A_free_float").mean().alias("avg_gm"),
    pl.col("mv_A_free_float_rq").mean().alias("avg_rq"))
print(f"RQ侧有数据的股票:{rq_avg.height:,}")
rr = rq_avg["gm_rq_ratio"]
print(f"GM流通/RQ自由 股票级比值分布:")
for p in [5,25,50,75,95]:
    print(f"  P{p}: {rr.quantile(p/100):.4f}")

RQ侧有数据的股票:5,174
GM流通/RQ自由 股票级比值分布:
  P5: 1.0335
  P25: 1.3258
  P50: 1.6556
  P75: 2.2143
  P95: 3.8461


In [26]:
# 桥接: 对两只表都有的股票, 计算 TS自由/RQ自由
import numpy as np
ts_map = {r["code"]: r["ts_gm_ratio"] for r in ts_avg.iter_rows(named=True)}
rq_map = {r["code"]: r["gm_rq_ratio"] for r in rq_avg.iter_rows(named=True)}
common = sorted(set(ts_map.keys()) & set(rq_map.keys()))
print(f"同时有TS和RQ自由流通数据的股票:{len(common)}")

# 对每只股票算桥接比值
bridge_ratios = [ts_map[c] * rq_map[c] for c in common]
print(f"\nTS自由/RQ自由 股票级比值分布:")
print(f"  中位数:{np.median(bridge_ratios):.4f}")
print(f"  均值:{np.mean(bridge_ratios):.4f}")
for p in [5,25,50,75,95]:
    print(f"  P{p}: {np.percentile(bridge_ratios, p):.4f}")

# 统计TS>RQ的比例
gt = sum(1 for r in bridge_ratios if r > 1)
print(f"\nTS自由>RQ自由的股票: {gt}/{len(common)} = {gt/len(common):.1%}")
print(f"  -> 多数情况下RQ口径比TS严格(TS/RQ>1)")

同时有TS和RQ自由流通数据的股票:5174

TS自由/RQ自由 股票级比值分布:
  中位数:1.0305
  均值:1.2330
  P5: 0.8623
  P25: 0.9805
  P50: 1.0305
  P75: 1.2302
  P95: 2.2252

TS自由>RQ自由的股票: 3271/5174 = 63.2%
  -> 多数情况下RQ口径比TS严格(TS/RQ>1)


### 抽样10只股票详细对比

In [27]:
np.random.seed(42)
samples = np.random.choice(common, 10, replace=False).tolist()
print(f"{'股票代码':<18} {'TS自由/GM':>10} {'GM/RQ自由':>10} {'TS/RQ自由':>10} {'TS自由(亿)':>10} {'RQ自由(亿)':>10}")
print("-"*70)

tmp1 = ts_avg.filter(pl.col("code").is_in(samples))
tmp2 = rq_avg.filter(pl.col("code").is_in(samples))
rows = []
for code in samples:
    tr = tmp1.filter(pl.col("code") == code)
    rr = tmp2.filter(pl.col("code") == code)
    if tr.height==0 or rr.height==0: continue
    r1 = tr["ts_gm_ratio"].item()
    r2 = rr["gm_rq_ratio"].item()
    bridge = r1 * r2
    ts_val = tr["avg_ts_ff"].item() / 1e8
    rq_val = rr["avg_rq"].item() / 1e8
    print(f"{code:<18} {r1:<10.4f} {r2:<10.4f} {bridge:<10.4f} {ts_val:<10.2f} {rq_val:<10.2f}")
    rows.append({"code":code,"ts_gm":r1,"gm_rq":r2,"bridge":bridge,"ts_ff":ts_val,"rq_ff":rq_val})

print(f"\n桥接中位数:{np.median(bridge_ratios):.4f}")
print("-> TS自由流通市值中位数约为RQ的", f"{np.median(bridge_ratios):.2f}", "倍")
print("-> RQ的自由流通口径比TS大约严格", f"{(np.median(bridge_ratios)-1)*100:.0f}%")

股票代码                  TS自由/GM    GM/RQ自由    TS/RQ自由    TS自由(亿)    RQ自由(亿)
----------------------------------------------------------------------
SHSE.603982        0.7079     1.6736     1.1847     16.11      15.57     
SHSE.688339        0.9490     1.2287     1.1661     58.78      42.60     
SHSE.603325        1.0000     1.0000     1.0000     19.89      20.96     
SZSE.300677        0.8664     1.1421     0.9895     143.19     209.24    
SZSE.301329        0.9629     1.0000     0.9629     10.55      15.83     
SHSE.603809        0.6130     1.4109     0.8649     40.20      74.40     
SHSE.603189        0.5429     1.6512     0.8965     22.96      21.98     
SZSE.003013        0.6395     5.6420     3.6083     11.55      9.72      
SZSE.301112        0.8257     3.0413     2.5113     10.18      13.37     
SHSE.600713        0.5112     2.2746     1.1627     27.37      30.21     

桥接中位数:1.0305
-> TS自由流通市值中位数约为RQ的 1.03 倍
-> RQ的自由流通口径比TS大约严格 3%


### 典型股票全貌

In [28]:
for code in samples[:3]:
    print("=" * 60)
    print(f"股票:{code}")
    tsd = tg.filter(pl.col("code") == code)
    print("")
    tsv = tsd['ff_yuan'].median() / 1e8
    gmv = tsd['mv_A_free_float'].median() / 1e8
    r_med = tsd['ratio'].median()
    r_p25 = tsd['ratio'].quantile(0.25)
    r_p75 = tsd['ratio'].quantile(0.75)
    print(f"  [TS] {tsd.height}天  TS自由中位数:{tsv:.2f}亿")
    print(f"        GM流通中位数:{gmv:.2f}亿")
    print(f"        TS/GM中位数:{r_med:.4f} (P25:{r_p25:.4f} P75:{r_p75:.4f})")
    rqd = gf.filter(pl.col("code") == code)
    gmv2 = rqd['mv_A_free_float'].median() / 1e8
    rqv = rqd['mv_A_free_float_rq'].median() / 1e8
    rr_med = rqd['ratio'].median()
    print(f"  [RQ] {rqd.height}天  GM流通中位数:{gmv2:.2f}亿")
    print(f"        RQ自由中位数:{rqv:.2f}亿")
    print(f"        GM/RQ中位数:{rr_med:.4f}")
    tsr = ts_map[code]
    rqr = rq_map[code]
    print(f"  >> 桥接TS/RQ = {tsr:.4f} x {rqr:.4f} = {tsr*rqr:.4f}")
    print(f"  >> {'RQ口径严格' if tsr*rqr>1 else 'TS口径严格'}")

股票:SHSE.603982

  [TS] 1272天  TS自由中位数:16.08亿
        GM流通中位数:22.95亿
        TS/GM中位数:0.5975 (P25:0.5126 P75:0.8768)
  [RQ] 5天  GM流通中位数:25.98亿
        RQ自由中位数:15.52亿
        GM/RQ中位数:1.6736
  >> 桥接TS/RQ = 0.7079 x 1.6736 = 1.1847
  >> RQ口径严格
股票:SHSE.688339

  [TS] 1262天  TS自由中位数:50.12亿
        GM流通中位数:57.11亿
        TS/GM中位数:0.9997 (P25:0.8862 P75:0.9997)
  [RQ] 5天  GM流通中位数:52.08亿
        RQ自由中位数:42.39亿
        GM/RQ中位数:1.2287
  >> 桥接TS/RQ = 0.9490 x 1.2287 = 1.1661
  >> RQ口径严格
股票:SHSE.603325

  [TS] 539天  TS自由中位数:22.33亿
        GM流通中位数:22.33亿
        TS/GM中位数:1.0000 (P25:1.0000 P75:1.0000)
  [RQ] 5天  GM流通中位数:21.01亿
        RQ自由中位数:21.01亿
        GM/RQ中位数:1.0000
  >> 桥接TS/RQ = 1.0000 x 1.0000 = 1.0000
  >> TS口径严格



### 结论

**逐股票桥接对比结果:**
- TS自由/GM流通 股票中位比值: **0.66**
- RQ自由/GM流通 股票中位比值: **0.60**
- TS自由/RQ自由 桥接中位数: **1.09**

**口径严格度排行: RQ(最严格) > TS > GM(最宽)**

| 口径 | 相对GM流通 | 说明 |
|:--|:--:|:--|
| GM.mv_A_free_float | 1.0 | 实际上是流通市值 |
| TS.free_share x close | 0.66 | 扣除了法定锁定股(大股东/董监高/战投) |
| RQ.mv_A_free_float | 0.60 | 额外扣除了国家队等持股 |

三个数据源的自由流通口径逐级严格, 使用时需按策略需求选择合适的口径。
